<a href="https://colab.research.google.com/github/mgkagori/dissertation-ecommerce-edt/blob/Data-Samples/Sample_1_MCAuley_Data_set_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U "datasets<3.0.0" "fsspec<=2025.3.0"

  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)


In [ ]:
from datasets import load_dataset

meta = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_Electronics",
    split="full", streaming=True, trust_remote_code=True
)

good_products = []
for m in meta:
    desc = m.get("description")
    if isinstance(desc, list) and len(desc) > 0 and len(" ".join(desc)) > 50:
        good_products.append(m)
    if len(good_products) >= 300:    # 300 well-described products
        break

print(f"Collected {len(good_products)} products with descriptions")
target_asins = set(p["parent_asin"] for p in good_products)

Collected 300 products with descriptions


In [ ]:
reviews = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Electronics",
    split="full", streaming=True, trust_remote_code=True
)

matched_reviews = []
checked = 0
for r in reviews:
    checked += 1
    if r.get("parent_asin") in target_asins:
        matched_reviews.append(r)
    if len(matched_reviews) >= 3000:   # enough reviews across those products
        break
    if checked >= 2_000_000:
        break

print(f"Collected {len(matched_reviews)} reviews for products")

Collected 877 reviews for products


In [ ]:
import pandas as pd
reviews_df = pd.DataFrame(matched_reviews)
meta_df = pd.DataFrame(good_products).rename(columns={
    "title": "product_title", "average_rating": "product_avg_rating"})

meta_cols = ["parent_asin", "product_title", "description",
             "features", "price", "product_avg_rating", "store"]
meta_subset = meta_df[[c for c in meta_cols if c in meta_df.columns]]

merged = reviews_df.merge(meta_subset, on="parent_asin", how="left")
print(f"Merged: {len(merged)} reviews across {merged['parent_asin'].nunique()} products")
print(merged[["parent_asin","rating","text","description"]].head())

Merged: 877 reviews across 110 products
  parent_asin  rating                                               text  \
0  B08X2DCF2B     5.0  She has a real library now, and they are showi...   
1  B016XL20UM     5.0  I received this as a free review sample. It ni...   
2  B075S8H5LY     5.0                                           Love it!   
3  B0067HY1EQ     5.0                                       Good filters   
4  B08X2DCF2B     5.0  I order these disks all the time and they cons...   

                                         description  
0  [Combining an exceptional inkjet hub-printable...  
1  [iColor 10 inch Laptop Carrying Bag Sleeve,Mad...  
2  [The Fujifilm Instax Mini 8 Instant Film Camer...  
3  [Product Description, This kit consists of a: ...  
4  [Combining an exceptional inkjet hub-printable...  


In [ ]:
merged.to_parquet('/content/drive/MyDrive/dissertation/electronics_merged_clean.parquet')

In [ ]:
merged.to_csv('/content/drive/MyDrive/dissertation/electronics_merged_preview.csv', index=False)
print("CSV saved")

CSV saved


In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_parquet('/content/drive/MyDrive/dissertation/electronics_merged_clean.parquet')

print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
print(df.columns.tolist())
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows: 877, Columns: 16
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'product_title', 'description', 'features', 'price', 'product_avg_rating', 'store']


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,description,features,price,product_avg_rating,store
0,5.0,My mother uses these to back up video files sh...,"She has a real library now, and they are showi...",[],B000A0CV9S,B08X2DCF2B,AHOEABHRAFWXIT4JZ5MKJ3FMASGA,1504256299168,1,True,Verbatim 4.7GB Up to 16X Datalifeplus White In...,[Combining an exceptional inkjet hub-printable...,[50 high-grade non-rewritable DVD-R inkjet pri...,26.94,4.6,Verbatim
1,5.0,Decent construction and great for my Tab 3 10....,I received this as a free review sample. It ni...,[],B019838N6C,B016XL20UM,AG3S4FROO422V5KP7DJCBXVUQLJQ,1463932849000,2,False,ICOLOR Lovely Kitty 9.7 10 Inch Laptop Carryin...,"[iColor 10 inch Laptop Carrying Bag Sleeve,Mad...","[iColor 10 inch Laptop Carrying Bag Sleeve,Mad...",10.88,4.7,icolor
2,5.0,Five Stars,Love it!,[],B075S8H5LY,B075S8H5LY,AF5JSPANM6KY5VZIYWQFNDSQB7XA,1513651627172,0,True,Fujifilm Instax Mini 8 Instant Film Camera (Po...,[The Fujifilm Instax Mini 8 Instant Film Camer...,"[New slimmer and lighter body, Works with Fuji...",None,4.5,Fujifilm
3,5.0,Five Stars,Good filters,[],B00004ZCL0,B0067HY1EQ,AE7N4WK7OYMAXKXODT2MSG7H4FPA,1436629413000,0,True,Tiffen 77mm Photo Essentials Kit with UV Prote...,"[Product Description, This kit consists of a: ...","[UV protector, Circular polarizer, 77mm diamet...",89.99,4.6,Tiffen
4,5.0,Does the job,I order these disks all the time and they cons...,[],B0007M0VXW,B08X2DCF2B,AHJOKJQEYWRJQSWYGDWUFLUPQUUQ,1398833931000,2,True,Verbatim 4.7GB Up to 16X Datalifeplus White In...,[Combining an exceptional inkjet hub-printable...,[50 high-grade non-rewritable DVD-R inkjet pri...,26.94,4.6,Verbatim


In [ ]:
# How many reviews does each product have?
review_counts = merged.groupby('parent_asin').size()
print(review_counts.describe())

# How many products have at least 20 reviews?
print(f"Products with 20+ reviews: {(review_counts >= 20).sum()}")

count    110.000000
mean       7.972727
std       17.150419
min        1.000000
25%        1.000000
50%        2.000000
75%        6.000000
max      120.000000
dtype: float64
Products with 20+ reviews: 11
